In [4]:
#installing necessary packages
#pandas,azure-storage-blob,dotenv

In [5]:
# importing necessary libraries
import pandas as pd
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
import os

In [6]:
#Extraction
try:
    data = pd.read_csv("zipco_transaction.csv")
    print("Data extracted successfully!")
except Exception as e:
    print(f"data extraction failed: {e}")

Data extracted successfully!


In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Date                    1005 non-null   object 
 1   ProductName             1005 non-null   object 
 2   Quantity                1005 non-null   int64  
 3   UnitPrice               1005 non-null   float64
 4   StoreLocation           1005 non-null   object 
 5   PaymentType             1005 non-null   object 
 6   PromotionApplied        1005 non-null   bool   
 7   Weather                 1005 non-null   object 
 8   Temperature             904 non-null    float64
 9   StaffPerformanceRating  1005 non-null   object 
 10  CustomerFeedback        905 non-null    object 
 11  DeliveryTime_min        1005 non-null   int64  
 12  OrderType               1005 non-null   object 
 13  CustomerName            1005 non-null   object 
 14  CustomerAddress         1005 non-null   

In [8]:
#Data Cleaning and Transformation
#Remove duplicates
data.drop_duplicates(inplace=True)

In [9]:
#handle missing values fill missing string/object values with 'unknown'
string_columns = data.select_dtypes(include=['object']).columns
for col in string_columns:
    data[col] = data[col].fillna('unknown')

#handle missing values fill missing float/int  values with 'mean/median'
float_columns = data.select_dtypes(include=['float64', 'int64']).columns
for col in float_columns:
    data[col] = data[col].fillna(data[col].mean())

In [10]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Date                    1000 non-null   object 
 1   ProductName             1000 non-null   object 
 2   Quantity                1000 non-null   int64  
 3   UnitPrice               1000 non-null   float64
 4   StoreLocation           1000 non-null   object 
 5   PaymentType             1000 non-null   object 
 6   PromotionApplied        1000 non-null   bool   
 7   Weather                 1000 non-null   object 
 8   Temperature             1000 non-null   float64
 9   StaffPerformanceRating  1000 non-null   object 
 10  CustomerFeedback        1000 non-null   object 
 11  DeliveryTime_min        1000 non-null   int64  
 12  OrderType               1000 non-null   object 
 13  CustomerName            1000 non-null   object 
 14  CustomerAddress         1000 non-null   object

In [11]:
data.columns

Index(['Date', 'ProductName', 'Quantity', 'UnitPrice', 'StoreLocation',
       'PaymentType', 'PromotionApplied', 'Weather', 'Temperature',
       'StaffPerformanceRating', 'CustomerFeedback', 'DeliveryTime_min',
       'OrderType', 'CustomerName', 'CustomerAddress', 'Customer_PhoneNumber',
       'CustomerEmail', 'Staff_Name', 'Staff_Email', 'DayOfWeek',
       'TotalSales'],
      dtype='object')

In [23]:
# create the product table
products = data[['ProductName']].drop_duplicates().reset_index(drop=True)
products.index.name = 'product_id'
products = products.reset_index()

In [24]:
products.head()


,product_id,ProductName
0,0,Vanilla Cake
1,1,Red Velvet Cake
2,2,Chocolate Cake
3,3,Carrot Cake
4,4,Pizza Pepperoni


In [21]:
#customers table
customers = data[['CustomerName', 'CustomerAddress', 'Customer_PhoneNumber','CustomerEmail']].drop_duplicates().reset_index(drop=True)
customers.index.name = 'customer_id'
customers = customers.reset_index()

In [22]:
customers.head()

,customer_id,CustomerName,CustomerAddress,Customer_PhoneNumber,CustomerEmail
0,0,William Adams,"9851 David Green\r\nTonyaburgh, VA 02853",(916)427-7276x861,lisa00@example.net
1,1,Anthony Wiggins,"24682 Holly Stravenue\r\nMooreville, NH 13901",769.318.4373,michellefernandez@example.com
2,2,Ashley Duke,10184 Washington Trace Apt. 679\r\nEast Brandi...,702.520.3286,cooperwilliam@example.com
3,3,Brandon Taylor,"87194 Jeff Rue\r\nMitchellbury, CA 50463",622-527-9530,fsilva@example.net
4,4,Brittany Watkins,"850 Julia Groves\r\nHartview, WI 95954",759-517-8359,petersjoseph@example.net


In [25]:
#create staff table
staff = data[['Staff_Name', 'Staff_Email']].drop_duplicates().reset_index(drop=True)
staff.index.name = 'staff_id'
staff = staff.reset_index()

In [26]:
staff.head()

,staff_id,Staff_Name,Staff_Email
0,0,John Bridges,pdavidson@example.com
1,1,Sarah Bentley,ajohnson@example.net
2,2,Connie Cervantes,michele29@example.net
3,3,Jessica Stewart,xwilson@example.org
4,4,Cheryl Carpenter,christine96@example.org


In [30]:
#create transaction table
transactions = data.merge(products, on='ProductName', how='left') \
                   .merge(customers, on=['CustomerName', 'CustomerAddress', 'Customer_PhoneNumber','CustomerEmail'], how='left') \
                   .merge(staff, on=['Staff_Name', 'Staff_Email'], how='left')
transactions.index.name = 'transaction_id'
transactions = transactions.reset_index() \
                            [['Date','transaction_id', 'product_id', 'customer_id', 'staff_id', 'Quantity', 'UnitPrice', 'StoreLocation', \
                              'PaymentType', 'PromotionApplied', 'Weather', 'Temperature', 'StaffPerformanceRating', 'CustomerFeedback', \
                              'DeliveryTime_min','OrderType',  'DayOfWeek','TotalSales']]
transactions['Date'] = pd.to_datetime(transactions['Date'])



In [37]:
data.to_csv('data/cleaned_data.csv', index=False)
products.to_csv('data/products.csv', index=False)
customers.to_csv('data/customers.csv', index=False)
staff.to_csv('data/staff.csv', index=False)
transactions.to_csv('data/transactions.csv', index=False)

In [31]:
transactions.head()
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    1000 non-null   datetime64[ns]
 1   transaction_id          1000 non-null   int64         
 2   product_id              1000 non-null   int64         
 3   customer_id             1000 non-null   int64         
 4   staff_id                1000 non-null   int64         
 5   Quantity                1000 non-null   int64         
 6   UnitPrice               1000 non-null   float64       
 7   StoreLocation           1000 non-null   object        
 8   PaymentType             1000 non-null   object        
 9   PromotionApplied        1000 non-null   bool          
 10  Weather                 1000 non-null   object        
 11  Temperature             1000 non-null   float64       
 12  StaffPerformanceRating  1000 non-null   object   